# 自動化 Face LoRA 生產線 (Google Colab 執行版)

這個筆記本將幫助您在 Google Colab 上自動執行 Face LoRA 生產線。
請確保您已經將整個專案資料夾上傳到了 Google Drive 的 `MyDrive/Face_LoRA_Pipeline` 目錄下。

## Training


In [ ]:
# 1. 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. 切換到專案工作目錄
import os

PROJECT_DIR = '/content/drive/MyDrive/Face_LoRA_Pipeline'

if not os.path.exists(PROJECT_DIR):
    print(f"錯誤：找不到路徑 {PROJECT_DIR}")
    print("請確保您已將專案上傳至 Google Drive 的正確位置。")
else:
    os.chdir(PROJECT_DIR)
    print(f"成功切換至工作目錄: {os.getcwd()}")

In [ ]:
# 3. 下載並安裝 kohya_ss 訓練環境
# 將 kohya_ss 腳本直接下載到 Google Drive 中，這樣下次就不用重新下載腳本了
import os
KOHYA_DIR = os.path.join(PROJECT_DIR, "kohya_ss")

if not os.path.exists(KOHYA_DIR):
    print("尚未下載 kohya_ss，開始從 GitHub 複製...")
    !git clone --recursive https://github.com/bmaltais/kohya_ss.git {KOHYA_DIR}
else:
    print("已在 Google Drive 中找到 kohya_ss 腳本，跳過下載步驟。")

# 切換至 kohya_ss 目錄
%cd {KOHYA_DIR}

# 確保子模組 (如 sd-scripts) 已正確初始化並更新
!git submodule update --init --recursive

# 【重要說明】：即使腳本存在 Drive 中，但 Colab 每次啟動都是一台全新的虛擬機。
# 因此 Python 套件必須「每次」重新安裝，否則會報錯找不到套件。
!pip install -r requirements.txt
!pip install accelerate transformers diffusers

# 安裝專案本身的依賴套件 (OpenCV, InsightFace 等)
!pip install opencv-python-headless insightface

# 切換回專案目錄
%cd {PROJECT_DIR}

In [ ]:
# 4. 執行全自動生產線主程式
# (請確認已經修改了 src/training/kohya_runner.py 移除了 Mock)
!python main.py

## 5. 獨立評分測試 

In [ ]:
#@title 5. 獨立評分測試 (Standalone Evaluation Testing)
# 這個區塊完全獨立於上面的流程，只要您的模型已經訓練好，就可以直接單獨執行這裡來產圖並查看評分。

# 5.1 安裝必要的套件 (獨立安裝確保環境乾淨)
from google.colab import drive
drive.mount('/content/drive')

!pip install -q diffusers==0.27.2 peft==0.10.0 transformers==4.40.0 accelerate==0.30.0 insightface onnxruntime-gpu huggingface-hub==0.25.2

import os
import torch
import cv2
import numpy as np
import insightface
from diffusers import StableDiffusionPipeline
from google.colab.patches import cv2_imshow

# 5.2 定義變數與路徑
FACE_ID = "TZUYU子瑜"  # ⚠️請替換為您剛剛訓練的臉部名稱
PROJECT_DIR = "/content/drive/MyDrive/app/AI/Lora/Face_LoRA_Pipeline"
MODEL_PATH = f"{PROJECT_DIR}/output/models/{FACE_ID}.safetensors"
BASE_MODEL_ID = "runwayml/stable-diffusion-v1-5" # 基底模型
ORIGINAL_IMAGES_DIR = f"{PROJECT_DIR}/training_data/{FACE_ID}"
EVAL_OUTPUT_DIR = f"{PROJECT_DIR}/output/eval/{FACE_ID}_standalone"

PROMPT = f"A portrait of {FACE_ID}, raw photo, highly detailed, 8k uhd, dslr"
NEGATIVE_PROMPT = "blurry, out of focus, disfigured, low quality, bad anatomy"
NUM_IMAGES = 10

os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

# 5.3 載入產圖模型 (Diffusers)
print("載入產圖模型中...")
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"找不到您的 LoRA 模型：{MODEL_PATH}，請確認檔名與路徑是否正確！")

pipe = StableDiffusionPipeline.from_pretrained(BASE_MODEL_ID, torch_dtype=dtype, safety_checker=None).to(device)
lora_dir = os.path.dirname(MODEL_PATH)
lora_file = os.path.basename(MODEL_PATH)
pipe.load_lora_weights(lora_dir, weight_name=lora_file)

# 5.4 開始產圖
generated_paths = []
print(f"開始產生 {NUM_IMAGES} 張測試圖片...")
for i in range(NUM_IMAGES):
    out_path = os.path.join(EVAL_OUTPUT_DIR, f"{FACE_ID}_standalone_{i}.jpg")
    img = pipe(prompt=PROMPT, negative_prompt=NEGATIVE_PROMPT, num_inference_steps=30, guidance_scale=7.5).images[0]
    img.save(out_path, format="JPEG", quality=95)
    generated_paths.append(out_path)
    print(f"已儲存: {out_path}")

del pipe
torch.cuda.empty_cache()

# 5.5 載入評分模型 (InsightFace)
print("\n載入 InsightFace 評分模型中...")
app = insightface.app.FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(640, 640))

def get_embedding(img_path):
    img = cv2.imread(img_path)
    if img is None: return None
    faces = app.get(img)
    return faces[0].normed_embedding if faces else None

# 5.6 計算相似度
print("\n開始計算相似度...")
original_images = [os.path.join(ORIGINAL_IMAGES_DIR, f) for f in os.listdir(ORIGINAL_IMAGES_DIR) if f.lower().endswith((".png", ".jpg", ".jpeg"))]

total_score = 0
comparisons = 0

for gen_path in generated_paths:
    gen_emb = get_embedding(gen_path)
    if gen_emb is None:
        print(f"\n--- ⚠️ 產出圖片 {os.path.basename(gen_path)}: 未偵測到人臉或圖片損壞，跳過評估 ---")
        continue
    
    print(f"\n--- 評估產出圖片: {os.path.basename(gen_path)} ---")
    display_img = cv2.imread(gen_path)
    display_img = cv2.resize(display_img, (256, 256))
    cv2_imshow(display_img)
    
    for orig_path in original_images:
        orig_emb = get_embedding(orig_path)
        if orig_emb is None: continue
        
        score = np.dot(gen_emb, orig_emb)
        total_score += score
        comparisons += 1

if comparisons > 0:
    avg_score = (total_score / comparisons) * 100
    print(f"\n======================================")
    print(f"最終平均相似度評分: {avg_score:.2f}%")
    print(f"======================================")
else:
    print("無法計算相似度，可能是圖片中未偵測到人臉。")








